# SAM 3 for insects — interactive inference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adambasha0/SAM3-for-Insects-segementation/blob/main/docs/sam3_insect_colab.ipynb)

Detection and instance segmentation of terrestrial arthropods with a **fine-tuned SAM 3**
(`flatbug_medium_ft`, epoch 18), wrapped in flatbug-style **pyramid tiling** so that small
insects in large trap photographs are still found.

Upload an image, press **Predict**, and download the predictions as COCO JSON — masks as
polygons, boxes as `[x, y, w, h]`, one confidence per detection.

---

### Before you start: turn the GPU on

**Runtime → Change runtime type → Hardware accelerator: GPU (T4 is enough) → Save**

On CPU the model loads but a single image takes many minutes, so the GPU is not optional in practice.

### What each step does

| Step | Time | What happens |
|---|---|---|
| 1 · Setup | ~2 min | Clone the repository and install dependencies |
| 2 · Weights | ~3–6 min | Download the 3.14 GB checkpoint and verify its checksum |
| 3 · Load | ~1 min | Build SAM 3 and load the fine-tuned weights onto the GPU |
| 4 · App | instant | Launch the upload-and-predict interface |
| 5 · Batch | optional | Run a whole folder (e.g. from Google Drive) to one COCO file |

Steps 1–3 only need to run once per Colab session.

## Step 1 · Setup

In [ ]:
# @title Clone the repository and install dependencies { display-mode: "form" }
REPO_URL = "https://github.com/adambasha0/SAM3-for-Insects-segementation.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}

import os, subprocess, sys

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")


def sh(cmd, **kwargs):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, **kwargs)


if not os.path.isdir(REPO_DIR):
    # A shallow clone skips the history; the clone is only a few MB because
    # the weights are fetched separately in step 2.
    sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])
else:
    print(f"{REPO_DIR} already present — skipping clone.")

os.chdir(REPO_DIR)

# --no-deps: the upstream SAM 3 metadata pins numpy==1.26, and downgrading
# numpy in Colab forces a runtime restart for no benefit. The dependencies we
# actually need are installed explicitly below.
sh([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
sh([sys.executable, "-m", "pip", "install", "-q",
    "gradio>=4", "timm>=1.0.17", "ftfy", "regex", "iopath>=0.1.10",
    "opencv-python-headless", "pyyaml", "huggingface_hub"])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch

print(f"\ntorch {torch.__version__} · CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("NO GPU — set Runtime → Change runtime type → GPU, then re-run this cell.")
import sam3_insect

print(f"sam3_insect {sam3_insect.__version__} ready.")

## Step 2 · Get the fine-tuned weights

The checkpoint is **3.14 GB of fp32 weights**, and GitHub caps a single file at 2 GB. So it
ships as **two ~1.57 GiB parts** that this notebook concatenates back into one `.pt` and
checks against a SHA-256 — a truncated download fails loudly instead of quietly loading
garbage weights.

Pick a source:

| `WEIGHTS_SOURCE` | Where it reads from | Notes |
|---|---|---|
| `release` | Two part files attached to the GitHub Release | **Default.** No auth, no rate limit |
| `hf` | A HuggingFace Hub mirror, if you published one | One file, resumable — set `hf_repo=` in the code |
| `local` | A path you already have | For a `.pt` you uploaded to `/content` |
| `drive` | Google Drive | Mounts Drive, then reads `LOCAL_PATH` — best if you rerun often |

All four end at the same file. `release` and `hf` are checksum-verified; `local` and `drive`
are used as-is, which also lets you point at the full 9.39 GB training checkpoint — the
loader drops the optimizer state for you.

Downloading takes a few minutes. If you expect to rerun this notebook, copy the finished
`.pt` to your Drive once and use `drive` afterwards.


In [ ]:
# @title Download / locate the checkpoint { display-mode: "form" }
WEIGHTS_SOURCE = "release"  # @param ["release", "hf", "local", "drive"]
LOCAL_PATH = ""  # @param {type:"string"}
KEEP_PARTS = False  # @param {type:"boolean"}

from sam3_insect import resolve_checkpoint

source = WEIGHTS_SOURCE
path = LOCAL_PATH or None

if source == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    if not path:
        raise ValueError(
            'With WEIGHTS_SOURCE="drive", set LOCAL_PATH to the checkpoint inside your '
            'Drive, e.g. /content/drive/MyDrive/checkpoint_18_inference.pt'
        )
    source = "local"

CHECKPOINT = resolve_checkpoint(
    source,
    path=path,
    cache_dir="/content/weights",
    keep_parts=KEEP_PARTS,
)
print(f"\nCheckpoint ready: {CHECKPOINT} ({CHECKPOINT.stat().st_size / 2**30:.2f} GiB)")

### Reassembling the parts yourself

The cell above does it for you, but the parts are a plain byte-wise split of an ordinary
file — you never depend on this repository's tooling to recover your weights:

```bash
# after downloading both parts
cat checkpoint_18_inference.pt.part-* > checkpoint_18_inference.pt
sha256sum -c checkpoint_18_inference.pt.sha256

# or let the packaged script download, join and verify in one step
./models/checkpoint_18_inference/fetch_weights.sh
```

```python
# or in Python
from sam3_insect import join_parts, INFERENCE_CKPT_SHA256
join_parts(["…part-00", "…part-01"], "checkpoint_18_inference.pt",
           expected_sha256=INFERENCE_CKPT_SHA256)
```


## Step 3 · Load the model

This builds the SAM 3 architecture and loads the fine-tuned weights into it. Nothing is
downloaded from the gated `facebook/sam3` HuggingFace repo: the fine-tuned checkpoint
contains every weight the model needs, so **no HuggingFace token and no licence acceptance
are required**.

The cell finishes with a smoke test on a bundled example, so you can see predictions even
if you skip the interactive app.

In [ ]:
import time

from sam3_insect import InsectPredictor

t0 = time.time()
predictor = InsectPredictor(CHECKPOINT)
print(f"Loaded in {time.time() - t0:.0f}s")

# --- smoke test -----------------------------------------------------------
EXAMPLES = [
    "docs/examples/example_petri_dish_ALUS.jpg",
    "docs/examples/example_single_insect_ArTaxOr.jpg",
    "docs/examples/example_pitfall_trap_UBC.jpg",
]

t0 = time.time()
result = predictor.predict(EXAMPLES[0])
kept = [a for a in result.annotations if a["score"] >= 0.4]
print(f"{result.file_name}: {len(kept)} detections at conf>=0.4 "
      f"({len(result)} raw) in {time.time() - t0:.1f}s")

overview = predictor.render(result, score_threshold=0.4)
overview.thumbnail((900, 900))
overview

## Step 4 · The interactive app

Running this cell prints two links and embeds the interface below it. Use the public
`*.gradio.live` link if the embedded view is cramped; it stays alive as long as this
notebook keeps running.

Inference runs **once** per image at a low score threshold, and the **Confidence** slider
then re-filters those cached detections — so sweeping the slider is instant instead of
costing another pass over the image pyramid. The **Bundle (.zip)** download contains the
COCO JSON, the rendered overview and one PNG per detected insect.

In [ ]:
from sam3_insect.app import build_demo

demo = build_demo(predictor, examples=EXAMPLES)
demo.queue().launch(share=True, debug=False)

## Step 5 · Batch a whole folder (optional)

For more than a handful of images, skip the GUI and write one COCO file for the lot. Point
`INPUT_DIR` at a folder in `/content`, or mount Google Drive and point it there.

Expect roughly 10–60 s per image on a Colab T4, depending on how many megapixels the
pyramid has to cover.

In [ ]:
# @title Batch inference { display-mode: "form" }
INPUT_DIR = "docs/examples"  # @param {type:"string"}
OUTPUT_DIR = "/content/predictions"  # @param {type:"string"}
CONFIDENCE = 0.4  # @param {type:"slider", min:0.02, max:0.95, step:0.01}
MAX_IMAGES = 0  # @param {type:"integer"}
MOUNT_DRIVE = False  # @param {type:"boolean"}

import glob
from pathlib import Path

from sam3_insect import annotations_to_coco
from sam3_insect.coco import save_coco

if MOUNT_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

files = sorted(
    p
    for p in glob.glob(os.path.join(INPUT_DIR, "**", "*"), recursive=True)
    if p.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff"))
)
if MAX_IMAGES:
    files = files[:MAX_IMAGES]
print(f"{len(files)} image(s) under {INPUT_DIR}")

out_dir = Path(OUTPUT_DIR)
(out_dir / "overviews").mkdir(parents=True, exist_ok=True)

results = []
for index, image_path in enumerate(files, start=1):
    result = predictor.predict(image_path)
    result.annotations = [a for a in result.annotations if a["score"] >= CONFIDENCE]
    results.append(result)
    predictor.render(result).save(
        out_dir / "overviews" / f"overview_{Path(result.file_name).stem}.jpg", quality=95
    )
    print(f"  [{index}/{len(files)}] {result.file_name}: {len(result)} detections")

coco_path = save_coco(annotations_to_coco(results), out_dir / "coco_instances.json")
print(f"\nWrote {coco_path} — {sum(len(r) for r in results)} annotations total.")

# Zip everything up and offer it as a browser download.
import shutil

archive = shutil.make_archive("/content/sam3_insect_predictions", "zip", out_dir)
try:
    from google.colab import files as colab_files

    colab_files.download(archive)
except ImportError:
    print(f"Archive at {archive}")

## Notes

**Score thresholds.** The library runs at `SCORE_THRESHOLD = 0.02`. Anything below that is
almost entirely an artefact of how the decoder works: SAM 3 spends a fixed budget of 200
object queries in full on every tile and has no per-query "nothing here" output, so a tile
holding *N* insects yields ~200 detections above 0.005 and roughly *N* above 0.5 — at the
same recall. Real detections in the fine-tuning domain usually score above 0.8, which is why
the app filters its display at 0.4. **Fix a threshold before reporting a count.**

**EXIF orientation.** Images are loaded through `PIL.ImageOps.exif_transpose`, so a phone
photo is predicted in the orientation you see. Pass `cfg={"EXIF_TRANSPOSE": False}` to work
in raw sensor coordinates instead.

**Precision.** The weights are unmodified fp32. fp16 was tried and rejected — the text
projection holds values up to 9.58e18, far past fp16's 65504 ceiling, which turns those
weights into `inf`. Activations are autocast at runtime instead: bfloat16 where the GPU
supports it, float16 on older cards such as Colab's T4, which have no bfloat16 units.

**Tuning knobs.** Every entry in `sam3_insect.DEFAULT_CFG` can be overridden per predictor or
per call. The two worth reaching for: `MASK_THRESHOLD` (lower grows masks, try 0.3 if they
hug the specimen too tightly) and `SCALE_BEFORE` (pre-upscale so tiny insects span more model
pixels).

```python
predictor.predict("image.jpg", cfg={"MASK_THRESHOLD": 0.35, "SCALE_BEFORE": 1.5})
```

**Where to go next.** [`README.md`](https://github.com/adambasha0/SAM3-for-Insects-segementation)
covers the CLI and local use; [`MODEL_CARD.md`](https://github.com/adambasha0/SAM3-for-Insects-segementation/blob/main/MODEL_CARD.md)
covers what the model was trained on and where it fails.
